In [27]:
# !pip install requests pandas sentence-transformers hdbscan google-generativeai jupyter

In [28]:
# !pip install streamlit requests sentence-transformers hdbscan pandas numpy google-genai

In [29]:
# %pip install umap-learn

In [30]:
import umap
import hdbscan
from sentence_transformers import SentenceTransformer
import requests
import pandas as pd
import os
from google import genai
import numpy as np
from pydantic import BaseModel, Field
from google.genai.types import GenerateContentConfig

class TrendInsight(BaseModel):
    trend_name: str = Field(description="A catchy 2-to-4 word label for the trend.")
    key_ingredients_or_products: list[str] = Field(description="Specific products, ingredients, or tools explicitly mentioned in the posts.")
    consumer_pain_point: str = Field(description="The underlying problem or insecurity the consumers are trying to solve.")
    capitalization_strategy: str = Field(description="A 1-sentence idea on how a brand could capitalize on this specific trend. Make it directand actionable.")
    actionability_score: int = Field(description="A score from 1-10 on how easily a business could monetize this trend.")


# --- Credentials ---
BSKY_HANDLE = os.getenv("BSKY_HANDLE")
BSKY_APP_PASSWORD = os.getenv("BSKY_APP_PASSWORD")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")


In [31]:
def fetch_bluesky_posts(query, target_count):

    # 1. Create a session to get the auth token
    session_url = "https://bsky.social/xrpc/com.atproto.server.createSession"
    session_data = {"identifier": BSKY_HANDLE, "password": BSKY_APP_PASSWORD}
    session_resp = requests.post(session_url, json=session_data).json()
    
    if "accessJwt" not in session_resp:
        raise Exception(f"Failed to authenticate: {session_resp}")
        
    auth_token = session_resp["accessJwt"]
    headers = {"Authorization": f"Bearer {auth_token}"}
    
    # 2. Search for posts iteratively
    search_url = "https://bsky.social/xrpc/app.bsky.feed.searchPosts"
    
    posts_data = []
    cursor = None
    
    print(f"Fetching {target_count} posts for '{query}'...")
    while len(posts_data) < target_count:
        params = {"q": query, "limit": 100} # 100 is the max per request
        if cursor:
            params["cursor"] = cursor
            
        resp = requests.get(search_url, headers=headers, params=params).json()
        new_posts = resp.get("posts", [])
        
        if not new_posts:
            break # No more posts available
            
        for post in new_posts:
            # We extract the text, the timestamp, and the author
            posts_data.append({
                "text": post["record"]["text"],
                "created_at": post["record"]["createdAt"],
                "author": post["author"]["handle"],
                "replyCount": post.get("replyCount", 0),
                "repostCount": post.get("repostCount", 0),
                "likeCount": post.get("likeCount", 0),
                "quoteCount": post.get("quoteCount", 0)
                # "has_embed_link": has_embed,
                # "labels": labels
            })
            
        cursor = resp.get("cursor")
        if not cursor:
            break
            
    # Keep only the target amount and convert to a DataFrame
    df = pd.DataFrame(posts_data[:target_count])
    print(f"Successfully fetched {len(df)} posts.")
    return df

# Test
# df_posts = fetch_bluesky_posts("skincare", target_count=5000)
# df_posts.head()

In [32]:
def filter_spam_posts(df, threshold):
    """
    Calculates a spam score (0.0 to 1.0) based on engagement, duplication, 
    and text formatting, then filters out posts above the threshold.
    """
    # Create a copy to avoid SettingWithCopyWarning
    df_scored = df.copy()
    
    # Initialize score
    df_scored['spam_score'] = 0.0
    
    # 1. Duplication Penalty (Strongest signal)
    # Flag posts that share the exact same text (e.g., cross-posting bots)
    is_duplicate = df_scored.duplicated(subset=['text'], keep='first')
    df_scored.loc[is_duplicate, 'spam_score'] += 0.5
    
    # 2. Low Engagement Penalty
    # Summing up the engagement metrics you are already fetching
    df_scored['total_engagement'] = (
        df_scored['replyCount'] + 
        df_scored['repostCount'] + 
        df_scored['likeCount'] + 
        df_scored['quoteCount']
    )
    # Add a penalty if the post has completely zero engagement
    df_scored.loc[df_scored['total_engagement'] == 0, 'spam_score'] += 0.2
    
    # 3. Content Heuristics (Links & Hashtags)
    # Count occurrences using basic regex
    df_scored['hashtag_count'] = df_scored['text'].str.count(r'#\w+')
    df_scored['link_count'] = df_scored['text'].str.count(r'http[s]?://')
    
    # Penalize spammy text formatting
    df_scored.loc[df_scored['hashtag_count'] > 4, 'spam_score'] += 0.15
    df_scored.loc[df_scored['link_count'] >= 2, 'spam_score'] += 0.15
    
    # 4. Cap the maximum score at 1.0
    df_scored['spam_score'] = df_scored['spam_score'].clip(upper=1.0)
    
    # Filter the DataFrame based on the acceptable threshold
    initial_count = len(df_scored)
    df_filtered = df_scored[df_scored['spam_score'] < threshold].copy()
    filtered_count = len(df_filtered)
    
    print(f"Filtered out {initial_count - filtered_count} spam-likely posts.")
    
    # Clean up calculation columns before passing to the clustering phase
    df_filtered = df_filtered.drop(columns=['total_engagement', 'hashtag_count', 'link_count'])
    
    return df_filtered


# df_posts = fetch_bluesky_posts("skincare", target_count=5000)
# df_clean = filter_spam_posts(df_posts, threshold=0.6)
# df_clustered = cluster_social_posts(df_clean)

In [33]:

df_posts = fetch_bluesky_posts("toothpaste", target_count=5000)
df_clean = filter_spam_posts(df_posts, threshold=0.1)

df_posts



Fetching 5000 posts for 'toothpaste'...
Successfully fetched 5000 posts.
Filtered out 1644 spam-likely posts.


,text,created_at,author,replyCount,repostCount,likeCount,quoteCount
0,Toothpaste probably! *Keeps pushing*,2026-07-31T20:11:26.315Z,hmmmbear.bsky.social,1,0,1,0
1,If you did- would it be like a cat kneading do...,2026-07-31T20:10:15.199Z,audrascollective.bsky.social,1,0,0,0
2,hang on one sec. turn around. hold still! you ...,2026-07-31T18:59:45.025Z,stel.energy,1,0,12,0
3,You can’t put toothpaste back in the tube. Hal...,2026-07-31T18:33:49.716Z,bingeingforsoup.bsky.social,0,1,23,1
4,A successful writer friend bragged that “he” j...,2026-07-31T18:30:04.842Z,rantdog.bsky.social,0,0,1,0
...,...,...,...,...,...,...,...
4995,‘I would never have guessed it’: Unexpected ef...,2026-05-29T15:40:04+00:00,eu-science.bsky.social,0,1,1,0
4996,Jesus Christ. Absolutely nothing achieved. Exc...,2026-05-29T15:34:39.148Z,ccpa-tirp.bsky.social,0,0,1,0
4997,‘I would never have guessed it’: Unexpected ef...,2026-05-29T15:30:05+00:00,uk-news.bsky.social,0,0,0,0
4998,"If that's mint, don't let anybody (there's 100...",2026-05-29T14:59:23.555Z,pxljim.bsky.social,1,0,1,0


In [34]:
df_clean

,text,created_at,author,replyCount,repostCount,likeCount,quoteCount,spam_score
0,Toothpaste probably! *Keeps pushing*,2026-07-31T20:11:26.315Z,hmmmbear.bsky.social,1,0,1,0,0.0
1,If you did- would it be like a cat kneading do...,2026-07-31T20:10:15.199Z,audrascollective.bsky.social,1,0,0,0,0.0
2,hang on one sec. turn around. hold still! you ...,2026-07-31T18:59:45.025Z,stel.energy,1,0,12,0,0.0
3,You can’t put toothpaste back in the tube. Hal...,2026-07-31T18:33:49.716Z,bingeingforsoup.bsky.social,0,1,23,1,0.0
4,A successful writer friend bragged that “he” j...,2026-07-31T18:30:04.842Z,rantdog.bsky.social,0,0,1,0,0.0
...,...,...,...,...,...,...,...,...
4990,Mischa just had her first little toothbrushing...,2026-05-29T16:29:18.499Z,felixffern.bsky.social,1,0,1,0,0.0
4995,‘I would never have guessed it’: Unexpected ef...,2026-05-29T15:40:04+00:00,eu-science.bsky.social,0,1,1,0,0.0
4996,Jesus Christ. Absolutely nothing achieved. Exc...,2026-05-29T15:34:39.148Z,ccpa-tirp.bsky.social,0,0,1,0,0.0
4998,"If that's mint, don't let anybody (there's 100...",2026-05-29T14:59:23.555Z,pxljim.bsky.social,1,0,1,0,0.0


In [35]:
def cluster_social_posts(df, cluster_fraction, sample_fraction):
    print("Loading Sentence Transformer model...")
    model = SentenceTransformer('all-MiniLM-L6-v2') 
    
    print("Generating embeddings...")
    embeddings = model.encode(df['text'].tolist())
    
    print("Reducing dimensions with UMAP...")
    # Compress the 384 dimensions down to 5 to help HDBSCAN find density
    umap_model = umap.UMAP(
        n_neighbors=15, # Focuses on local neighborhood size (default is 15)
        n_components=5, # Reduce to 5 dimensions
        min_dist=0.0,   # How tightly to pack points together 
        metric='cosine',# Cosine works best for text embeddings
        random_state=42 # Ensure reproducible results
    )
    reduced_embeddings = umap_model.fit_transform(embeddings)
    
    print("Running HDBSCAN clustering...")
    
    # Calculate dynamic parameters based on DataFrame size
    total_posts = len(df)
    
    # Force the values to be integers, and set an absolute minimum floor (e.g., 5)
    # so small datasets don't end up with a min_cluster_size of 1.
    dynamic_min_cluster_size = max(5, int(total_posts * cluster_fraction))
    dynamic_min_samples = max(2, int(total_posts * sample_fraction))
    
    print(f"Dynamic Settings: min_cluster_size={dynamic_min_cluster_size}, min_samples={dynamic_min_samples}")
    
    print("Running HDBSCAN clustering...")
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=dynamic_min_cluster_size, 
        min_samples=dynamic_min_samples,      
        metric='euclidean'  
    )


    df['cluster_id'] = clusterer.fit_predict(reduced_embeddings)
    
    # -1 means "noise" (unclustered). Let's filter those out.
    clustered_df = df[df['cluster_id'] != -1]
    
    print(f"Found {len(clustered_df['cluster_id'].unique())} unique clusters.")
    return clustered_df
# Test
df_clustered = cluster_social_posts(df_clean, 0.005, 0.002)
print(df_clustered['cluster_id'].value_counts())

Loading Sentence Transformer model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Generating embeddings...
Reducing dimensions with UMAP...


c:\Users\JesusSantillanMinila\Documents\GitHub\social-media-trend-detector\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Running HDBSCAN clustering...
Dynamic Settings: min_cluster_size=16, min_samples=6
Running HDBSCAN clustering...
Found 42 unique clusters.
cluster_id
26    255
24    142
33    122
41    101
35     85
16     76
40     70
32     67
1      62
38     57
0      55
27     54
39     48
8      48
36     48
12     46
11     45
18     44
19     38
28     37
22     37
20     35
31     30
29     28
25     28
37     25
4      24
10     24
3      23
17     23
7      22
6      22
34     21
13     21
9      20
21     19
15     19
5      19
23     18
14     17
2      17
30     16
Name: count, dtype: int64


In [36]:
total_posts = len(df_clean)
total_posts

3356

In [37]:
# def extract_actionable_insights(df_clustered):
#     client = genai.Client(api_key=GEMINI_API_KEY)
    
#     # Create a dictionary to hold our rich insights
#     cluster_insights = {}
#     unique_clusters = df_clustered['cluster_id'].unique()
    
#     for cluster_id in unique_clusters:
#         # Increase the sample size slightly for better context
#         sample_posts = df_clustered[df_clustered['cluster_id'] == cluster_id]['text'].head(10).tolist()
#         posts_text = "\n- ".join(sample_posts)
        
#         prompt = f"""
#         You are an expert consumer trend analyst and product developer. 
#         Analyze the following social media posts that have been clustered together:
#         - {posts_text}
        
#         Extract the underlying trend and identify exactly how a business can capitalize on it.
#         """
        
#         # Enforce structured output via GenerateContentConfig
#         response = client.models.generate_content(
#             model='gemini-3.5-flash-lite', 
#             contents=prompt,
#             config=GenerateContentConfig(
#                 response_mime_type="application/json",
#                 response_schema=TrendInsight,
#             )
#         )
        
#         # Access the structured data safely using .parsed
#         insight = response.parsed
#         cluster_insights[cluster_id] = insight
        
#         print(f"Analyzed Cluster {cluster_id}: {insight.trend_name}. \nProduct: {insight.capitalization_strategy} (Score: {insight.actionability_score}/10)")
        
#     # Map the new structured data back to the dataframe
#     df_clustered['trend_name'] = df_clustered['cluster_id'].map(lambda x: cluster_insights[x].trend_name if x in cluster_insights else None)
#     df_clustered['key_products'] = df_clustered['cluster_id'].map(lambda x: ", ".join(cluster_insights[x].key_ingredients_or_products) if x in cluster_insights else None)
#     df_clustered['pain_point'] = df_clustered['cluster_id'].map(lambda x: cluster_insights[x].consumer_pain_point if x in cluster_insights else None)
#     df_clustered['strategy'] = df_clustered['cluster_id'].map(lambda x: cluster_insights[x].capitalization_strategy if x in cluster_insights else None)
#     df_clustered['actionability_score'] = df_clustered['cluster_id'].map(lambda x: cluster_insights[x].actionability_score if x in cluster_insights else None)
    
#     return df_clustered

In [38]:

# Test 
df_labeled = extract_actionable_insights(df_clustered)

Analyzed Cluster 24: J-Hope Toothpaste Crossover. 
Product: Partner with K-pop icons or major influencers to release limited-edition branded oral care products that tap into fandom collectibility. (Score: 7/10)
Analyzed Cluster 26: Toothpaste Sensitivity & Irritation. 
Product: Brands should formulate an ultra-gentle, allergen-free, and flavor-free toothpaste specifically designed for sensitive oral mucosa to prevent canker sores and allergic reactions. (Score: 8/10)
Analyzed Cluster 41: Irreversible Reality Acceptance. 
Product: Brands should launch marketing campaigns focused on forward-looking adaptation and coping products rather than attempting to reverse modern cultural or technological milestones. (Score: 6/10)
Analyzed Cluster 33: Non-Mint Oral Care. 
Product: Launch a diverse line of non-mint, dessert-inspired or fruit-flavored toothpastes marketed specifically for sensitive teeth and bedtime routines. (Score: 9/10)
Analyzed Cluster 19: Absurdist Toothpaste Humor. 
Product: La

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 15, model: gemini-3.5-flash-lite\nPlease retry in 9.52701956s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.5-fl